# Analisis Geospasial Kebakaran Hutan dan Lahan (Karhutla) di Indonesia

**Tujuan.** Notebook ini menyajikan alur analisis data spasial titik panas (*hotspot*) kebakaran hutan dan lahan di Indonesia, mulai dari pemuatan data, pembersihan data, hingga operasi geospasial menggunakan pustaka `pandas`, `geopandas`, dan `matplotlib`.

**Sumber data.**

| No | Sumber | Berkas | Keterangan |
|---|---|---|---|
| 1 | NASA FIRMS (VIIRS S-NPP, C2) | `DL_FIRE_SV-C2_839983/fire_archive_*.csv` dan `fire_nrt_*.csv` | Titik panas individual (lintang/bujur) — *archive* (terverifikasi) dan *near real-time* |
| 2 | SiPongi (KLHK) | `sebaran-hotspot.xlsx` | Sheet `Akumulasi` (rekap jumlah per kab/kota) dan `Detail` (titik per desa) |

**Struktur notebook.**
0. Persiapan lingkungan
1. *Data Loading*
2. *Data Cleaning*
3. **Tugas 1** — Eksplorasi data spasial dasar
4. **Tugas 2** — Transformasi proyeksi koordinat (CRS)
5. **Tugas 3** — Operasi geometri (*buffer* 500 m)
6. **Tugas 4** — Visualisasi tematik (*choropleth map*)
7. Kesimpulan

## 0. Persiapan Lingkungan

Sel berikut mengimpor pustaka yang dibutuhkan. Pembacaan berkas Excel memerlukan paket `openpyxl`; bila paket tersebut (atau `geopandas`) belum tersedia pada kernel yang aktif, sel ini akan memasangnya secara otomatis.

In [7]:
import sys
import subprocess
import importlib.util

# Pasang paket yang belum tersedia di kernel ini (openpyxl dibutuhkan untuk membaca Excel)
for paket in ["geopandas", "openpyxl"]:
    if importlib.util.find_spec(paket) is None:
        print(f"Memasang {paket} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paket])

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from shapely.geometry import box

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

print("pandas    :", pd.__version__)
print("geopandas :", gpd.__version__)

pandas    : 2.2.3
geopandas : 1.2.0


Selanjutnya didefinisikan lokasi berkas dan konstanta analisis. Folder data dicari otomatis dari daftar `KANDIDAT_DIR`; jika data disimpan di lokasi lain (mis. Google Drive), tambahkan path folder tersebut ke daftar itu.

In [8]:
# Mencari folder data secara otomatis (kernel Jupyter belum tentu berjalan di folder notebook)
KANDIDAT_DIR = [
    Path.cwd(),
    Path(r"C:\Users\User\Downloads\Analisis Data Spasial"),   # folder lokal
    Path("/content"),                                        # Google Colab
    Path("/content/drive/MyDrive/Analisis Data Spasial"),    # Google Drive (Colab)
]
BASE_DIR = next((d for d in KANDIDAT_DIR if (d / "sebaran-hotspot.xlsx").exists()), None)
if BASE_DIR is None:   # cari di subfolder (mis. hasil unggah/ekstrak di Colab)
    for akar in [Path.cwd(), Path("/content")]:
        if akar.exists():
            hasil = next(akar.rglob("sebaran-hotspot.xlsx"), None)
            if hasil is not None:
                BASE_DIR = hasil.parent
                break
if BASE_DIR is None:
    raise FileNotFoundError(
        "Folder data tidak ditemukan. Tambahkan lokasi folder yang berisi 'sebaran-hotspot.xlsx' "
        "dan 'DL_FIRE_SV-C2_839983' ke dalam daftar KANDIDAT_DIR.")
print("Folder data:", BASE_DIR.resolve())

DIR_NASA          = BASE_DIR / "DL_FIRE_SV-C2_839983"
PATH_NASA_ARCHIVE = DIR_NASA / "fire_archive_SV-C2_839983.csv"
PATH_NASA_NRT     = DIR_NASA / "fire_nrt_SV-C2_839983.csv"
PATH_SIPONGI      = BASE_DIR / "sebaran-hotspot.xlsx"

# Berkas spasial (GeoJSON dihasilkan di notebook ini; shapefile bersifat opsional)
PATH_GEOJSON       = BASE_DIR / "hotspot_nasa.geojson"
PATH_SHP_INDONESIA = BASE_DIR / "peta_indonesia.shp"

# Kotak batas (bounding box) wilayah Indonesia dalam derajat
BBOX_INDONESIA = {"lon_min": 94.0, "lon_max": 142.0, "lat_min": -11.5, "lat_max": 6.5}

# Bila folder NASA belum diekstrak tetapi berkas .zip tersedia, ekstrak otomatis
PATH_NASA_ZIP = BASE_DIR / "DL_FIRE_SV-C2_839983.zip"
if not PATH_NASA_ARCHIVE.exists() and PATH_NASA_ZIP.exists():
    import zipfile
    with zipfile.ZipFile(PATH_NASA_ZIP) as zf:
        zf.extractall(DIR_NASA)
    print("Berkas zip NASA diekstrak ke:", DIR_NASA)

for p in [PATH_NASA_ARCHIVE, PATH_NASA_NRT, PATH_SIPONGI, PATH_SHP_INDONESIA]:
    print(f"{'[ADA]  ' if p.exists() else '[TIDAK]'} {p}")

FileNotFoundError: Folder data tidak ditemukan. Tambahkan lokasi folder yang berisi 'sebaran-hotspot.xlsx' dan 'DL_FIRE_SV-C2_839983' ke dalam daftar KANDIDAT_DIR.

## 1. Data Loading

### 1.1 Data Titik Panas NASA FIRMS

Data NASA FIRMS terdiri atas dua berkas dengan struktur kolom yang hampir sama:
- **Archive** — data standar yang telah melalui proses kendali mutu (memiliki kolom tambahan `type`).
- **NRT (*Near Real-Time*)** — data terbaru yang belum masuk arsip.

Kedua berkas digabungkan menjadi satu DataFrame dengan penanda kolom `sumber`. Kolom `acq_time` dibaca sebagai teks agar angka nol di depan (mis. `0513`) tidak hilang.

In [ ]:
df_archive = pd.read_csv(PATH_NASA_ARCHIVE, dtype={"acq_time": str})
df_nrt     = pd.read_csv(PATH_NASA_NRT, dtype={"acq_time": str})

df_archive["sumber"] = "archive"
df_nrt["sumber"]     = "nrt"

df_nasa_raw = pd.concat([df_archive, df_nrt], ignore_index=True)

print(f"Archive : {len(df_archive):>9,} baris | {df_archive['acq_date'].min()} s.d. {df_archive['acq_date'].max()}")
print(f"NRT     : {len(df_nrt):>9,} baris | {df_nrt['acq_date'].min()} s.d. {df_nrt['acq_date'].max()}")
print(f"Gabungan: {len(df_nasa_raw):>9,} baris, {df_nasa_raw.shape[1]} kolom")
df_nasa_raw.head()

In [ ]:
print("Tipe data kolom:")
print(df_nasa_raw.dtypes.to_string())
print("\nJumlah nilai kosong per kolom:")
print(df_nasa_raw.isna().sum().to_string())

### 1.2 Data Rekapitulasi SiPongi

Berkas Excel SiPongi memiliki dua *sheet*:
- **`Akumulasi`** — jumlah hotspot (`Counter`) per Provinsi, Kabupaten/Kota, Satelit, dan tingkat kepercayaan (`Confidence`).
- **`Detail`** — daftar titik hotspot lengkap dengan wilayah administrasi (hingga desa), waktu, serta koordinat `Latitude`/`Longitude`.

In [ ]:
df_sipongi_akum   = pd.read_excel(PATH_SIPONGI, sheet_name="Akumulasi")
df_sipongi_detail = pd.read_excel(PATH_SIPONGI, sheet_name="Detail")

print(f"Sheet Akumulasi : {df_sipongi_akum.shape}")
print(f"Sheet Detail    : {df_sipongi_detail.shape}")
display(df_sipongi_akum.head())
display(df_sipongi_detail.head())

## 2. Data Cleaning

### 2.1 Pembersihan Data NASA FIRMS

Langkah pembersihan yang diterapkan:
1. Menghapus baris tanpa koordinat (`latitude`/`longitude` kosong).
2. Menghapus duplikat (koordinat, tanggal, jam, dan satelit yang identik).
3. Membatasi data pada kotak batas wilayah Indonesia.
4. Mempertahankan hanya deteksi **kebakaran vegetasi** (`type = 0`). Nilai `type` lain (gunung api, sumber panas statis/industri, lepas pantai) dibuang. Data NRT tidak memiliki kolom `type` sehingga tetap dipertahankan.
5. Membuang deteksi dengan tingkat kepercayaan rendah (`confidence = 'l'`), menyisakan `n` (*nominal*) dan `h` (*high*).
6. Menyeragamkan tipe data waktu (`acq_date`, `acq_time` → `datetime`) dan kolom `version`.

Jumlah baris pada setiap tahap dicatat agar proses pembersihan dapat ditelusuri.

In [ ]:
df = df_nasa_raw.copy()
log = [("0. Data mentah", len(df))]

# 1) Koordinat kosong
df = df.dropna(subset=["latitude", "longitude"])
log.append(("1. Hapus koordinat kosong", len(df)))

# 2) Duplikat
df = df.drop_duplicates(subset=["latitude", "longitude", "acq_date", "acq_time", "satellite"])
log.append(("2. Hapus duplikat", len(df)))

# 3) Batas wilayah Indonesia
b = BBOX_INDONESIA
df = df[df["longitude"].between(b["lon_min"], b["lon_max"]) &
        df["latitude"].between(b["lat_min"], b["lat_max"])]
log.append(("3. Filter bounding box Indonesia", len(df)))

# 4) Hanya kebakaran vegetasi (type 0); NRT tidak punya kolom type
df = df[df["type"].isna() | (df["type"] == 0)]
log.append(("4. Hanya kebakaran vegetasi (type 0)", len(df)))

# 5) Buang confidence rendah
df["confidence"] = df["confidence"].astype(str).str.lower().str.strip()
df = df[df["confidence"].isin(["n", "h"])]
log.append(("5. Buang confidence rendah ('l')", len(df)))

# 6) Penyeragaman tipe data
df["acq_time"]         = df["acq_time"].str.zfill(4)
df["acq_datetime_utc"] = pd.to_datetime(df["acq_date"] + " " + df["acq_time"], format="%Y-%m-%d %H%M")
df["acq_date"]         = pd.to_datetime(df["acq_date"])
df["bulan"]            = df["acq_date"].dt.to_period("M").astype(str)
df["version"]          = df["version"].astype(str)
df["confidence_label"] = df["confidence"].map({"n": "Nominal", "h": "High"})

df_nasa = df.reset_index(drop=True)

df_log = pd.DataFrame(log, columns=["Tahap", "Jumlah baris"])
df_log["Terbuang"] = (-df_log["Jumlah baris"].diff()).fillna(0).astype(int)
df_log

In [ ]:
hitung_bulan = (df_nasa.groupby(["bulan", "confidence_label"]).size()
                .unstack(fill_value=0)[["Nominal", "High"]])

ax = hitung_bulan.plot(kind="bar", stacked=True, figsize=(11, 4.5),
                       color={"Nominal": "#fdae61", "High": "#d7191c"})
ax.set_title("Jumlah Hotspot NASA FIRMS (VIIRS) per Bulan Setelah Pembersihan")
ax.set_xlabel("Bulan")
ax.set_ylabel("Jumlah hotspot")
ax.legend(title="Confidence")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 2.2 Pembersihan Data SiPongi

Langkah pembersihan:
1. Menyeragamkan nama kolom menjadi huruf kecil dengan pemisah garis bawah.
2. Membersihkan spasi berlebih pada kolom teks.
3. Mengonversi kolom numerik (`jumlah`, `latitude`, `longitude`) dan tanggal ke tipe data yang tepat.
4. Menghapus baris kosong dan duplikat.
5. Menyusun **rekap jumlah hotspot per provinsi** yang akan digunakan pada Tugas 4.

In [ ]:
# --- Sheet Akumulasi ---
df_akum = df_sipongi_akum.rename(columns={
    "Provinsi": "provinsi", "Kab Kota": "kab_kota", "Satelit": "satelit",
    "Confidence": "confidence", "Counter": "jumlah"})
for c in ["provinsi", "kab_kota", "satelit", "confidence"]:
    df_akum[c] = df_akum[c].astype(str).str.strip()
df_akum["jumlah"] = pd.to_numeric(df_akum["jumlah"], errors="coerce").fillna(0).astype(int)
df_akum = df_akum[df_akum["provinsi"].ne("") & df_akum["provinsi"].ne("nan")].drop_duplicates()

# --- Sheet Detail ---
df_detail = df_sipongi_detail.rename(columns=lambda c: c.strip().lower().replace(" ", "_"))
for c in ["provinsi", "kab_kota", "kecamatan", "desa", "satelit", "confidence"]:
    df_detail[c] = df_detail[c].astype(str).str.strip()
df_detail["latitude"]  = pd.to_numeric(df_detail["latitude"], errors="coerce")
df_detail["longitude"] = pd.to_numeric(df_detail["longitude"], errors="coerce")
df_detail["tanggal"]   = pd.to_datetime(df_detail["tanggal"], format="%d-%m-%Y", errors="coerce")
df_detail = df_detail.dropna(subset=["latitude", "longitude"]).drop_duplicates().reset_index(drop=True)

# --- Rekap per provinsi ---
df_prov_sipongi = (df_akum.groupby("provinsi", as_index=False)["jumlah"].sum()
                   .rename(columns={"jumlah": "jumlah_hotspot"})
                   .sort_values("jumlah_hotspot", ascending=False)
                   .reset_index(drop=True))

print(f"Total hotspot (Akumulasi) : {df_akum['jumlah'].sum():,}")
print(f"Total titik   (Detail)    : {len(df_detail):,}")
print(f"Jumlah provinsi           : {df_prov_sipongi['provinsi'].nunique()}")
df_prov_sipongi.head(10)

In [ ]:
top10 = df_prov_sipongi.head(10).sort_values("jumlah_hotspot")
warna = plt.cm.OrRd(np.linspace(0.35, 0.95, len(top10)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top10["provinsi"], top10["jumlah_hotspot"], color=warna, edgecolor="grey")
for i, v in enumerate(top10["jumlah_hotspot"]):
    ax.text(v + 2, i, f"{v:,}", va="center", fontsize=9)
ax.set_title("10 Provinsi dengan Jumlah Hotspot Tertinggi (SiPongi)")
ax.set_xlabel("Jumlah hotspot")
plt.tight_layout()
plt.show()

## 3. TUGAS 1 — Eksplorasi Data Spasial Dasar

Pada tahap ini data hotspot diperlakukan sebagai **data vektor bertipe titik** (*Point*). Langkah-langkahnya:
1. Mengonversi DataFrame NASA hasil pembersihan menjadi `GeoDataFrame`, lalu menyimpannya sebagai berkas `hotspot_nasa.geojson`. Untuk menjaga ukuran berkas tetap ringan, hanya hotspot dengan **confidence tinggi (`h`)** yang diekspor.
2. Membaca kembali berkas spasial tersebut dengan `gpd.read_file()` — fungsi yang sama juga dapat digunakan untuk membaca *shapefile* seperti `peta_indonesia.shp`.
3. Menampilkan 5 data teratas dengan `.head()`.
4. Menampilkan Sistem Koordinat Referensi (CRS) melalui atribut `.crs`.
5. Melakukan visualisasi sederhana dengan `.plot()`.

In [ ]:
# Persiapan: membuat berkas GeoJSON dari data NASA (confidence tinggi)
kolom_ekspor = ["latitude", "longitude", "acq_date", "acq_time", "satellite",
                "confidence", "brightness", "bright_t31", "frp", "daynight", "bulan"]
df_export = df_nasa.loc[df_nasa["confidence"] == "h", kolom_ekspor].copy()
df_export["acq_date"] = df_export["acq_date"].dt.strftime("%Y-%m-%d")

gdf_export = gpd.GeoDataFrame(
    df_export,
    geometry=gpd.points_from_xy(df_export["longitude"], df_export["latitude"]),
    crs="EPSG:4326",
)
gdf_export.to_file(PATH_GEOJSON, driver="GeoJSON")
print(f"Tersimpan: {PATH_GEOJSON} ({len(gdf_export):,} titik)")

In [ ]:
# 1) Membaca file spasial dengan gpd.read_file()
gdf_hotspot = gpd.read_file(PATH_GEOJSON)
print(type(gdf_hotspot))
print(f"Jumlah fitur : {len(gdf_hotspot):,}")
print(f"Tipe geometri: {gdf_hotspot.geom_type.unique().tolist()}")

# (Opsional / dummy code) Membaca shapefile batas wilayah Indonesia bila tersedia
if PATH_SHP_INDONESIA.exists():
    gdf_indonesia = gpd.read_file(PATH_SHP_INDONESIA)
    print(f"Shapefile batas wilayah dimuat: {len(gdf_indonesia)} poligon")
else:
    gdf_indonesia = None
    print("Catatan: 'peta_indonesia.shp' tidak ditemukan; peta latar batas wilayah dilewati.")

In [ ]:
# 2) Lima data teratas
gdf_hotspot.head()

In [ ]:
# 3) Informasi Sistem Koordinat Referensi (CRS)
crs = gdf_hotspot.crs
print("CRS             :", crs)
print("Nama            :", crs.name)
print("Kode EPSG       :", crs.to_epsg())
print("Geografis?      :", crs.is_geographic)
print("Satuan sumbu    :", [a.unit_name for a in crs.axis_info])
print("Batas (minx, miny, maxx, maxy):", np.round(gdf_hotspot.total_bounds, 4))

In [ ]:
# 4) Visualisasi spasial sederhana dengan .plot()
fig, ax = plt.subplots(figsize=(14, 6))
if gdf_indonesia is not None:
    gdf_indonesia.to_crs(gdf_hotspot.crs).plot(ax=ax, color="whitesmoke", edgecolor="grey", linewidth=0.4)

gdf_hotspot.plot(ax=ax, markersize=1.5, color="red", alpha=0.5, label="Hotspot (confidence tinggi)")

ax.set_title("Sebaran Titik Panas NASA FIRMS VIIRS di Indonesia — EPSG:4326")
ax.set_xlabel("Bujur (°)")
ax.set_ylabel("Lintang (°)")
ax.legend(loc="lower left", markerscale=6)
plt.tight_layout()
plt.show()

**Interpretasi Tugas 1.** Data dibaca sebagai `GeoDataFrame` dengan kolom `geometry` bertipe *Point*. CRS yang terbaca adalah **EPSG:4326 (WGS 84)**, yaitu sistem koordinat geografis dengan satuan **derajat**. Peta sebaran memperlihatkan pola spasial hotspot di wilayah Indonesia; konsentrasi titik dapat diamati secara visual pada pulau-pulau tertentu.

## 4. TUGAS 2 — Transformasi Proyeksi Koordinat (CRS)

Terdapat dua operasi CRS yang perlu dibedakan:

| Metode | Fungsi | Mengubah nilai koordinat? |
|---|---|---|
| `.set_crs(epsg=4326)` | **Menetapkan** metadata CRS pada data yang belum memiliki CRS | Tidak |
| `.to_crs(epsg=3857)` | **Memproyeksikan ulang** koordinat ke CRS lain | Ya |

Untuk mendemonstrasikan pengecekan CRS, titik-titik dari sheet `Detail` SiPongi dikonversi menjadi `GeoDataFrame` **tanpa** CRS, kemudian CRS ditetapkan bila belum ada. Setelah itu, kedua data diproyeksikan ke **EPSG:3857 (Web Mercator)** yang bersatuan **meter**.

In [ ]:
def pastikan_crs(gdf, epsg_default=4326, nama="GeoDataFrame"):
    """Menetapkan CRS default bila GeoDataFrame belum memiliki CRS."""
    if gdf.crs is None:
        print(f"[!] {nama}: CRS belum terdefinisi -> ditetapkan EPSG:{epsg_default}")
        gdf = gdf.set_crs(epsg=epsg_default)
    else:
        print(f"[OK] {nama}: CRS sudah terdefinisi -> {gdf.crs.to_string()}")
    return gdf

# GeoDataFrame titik SiPongi sengaja dibuat TANPA CRS
gdf_sipongi_pts = gpd.GeoDataFrame(
    df_detail, geometry=gpd.points_from_xy(df_detail["longitude"], df_detail["latitude"]))
print("CRS awal titik SiPongi:", gdf_sipongi_pts.crs)

gdf_sipongi_pts = pastikan_crs(gdf_sipongi_pts, nama="Titik SiPongi")
gdf_hotspot     = pastikan_crs(gdf_hotspot, nama="Hotspot NASA")

In [ ]:
# Transformasi ke CRS metrik Web Mercator
gdf_hotspot_3857 = gdf_hotspot.to_crs(epsg=3857)
gdf_sipongi_3857 = gdf_sipongi_pts.to_crs(epsg=3857)

print("CRS baru :", gdf_hotspot_3857.crs.name, "| satuan:", gdf_hotspot_3857.crs.axis_info[0].unit_name)
print("Batas EPSG:4326 :", np.round(gdf_hotspot.total_bounds, 4))
print("Batas EPSG:3857 :", np.round(gdf_hotspot_3857.total_bounds, 1))

pd.DataFrame({
    "EPSG:4326 (x, y) derajat": gdf_hotspot.geometry.head().apply(lambda g: f"({g.x:.5f}, {g.y:.5f})"),
    "EPSG:3857 (x, y) meter":   gdf_hotspot_3857.geometry.head().apply(lambda g: f"({g.x:,.1f}, {g.y:,.1f})"),
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

gdf_hotspot.plot(ax=axes[0], markersize=1, color="red", alpha=0.5)
gdf_sipongi_pts.plot(ax=axes[0], markersize=4, color="navy", alpha=0.7)
axes[0].set_title("Sebelum: EPSG:4326 (WGS 84, derajat)")
axes[0].set_xlabel("Bujur (°)"); axes[0].set_ylabel("Lintang (°)")

gdf_hotspot_3857.plot(ax=axes[1], markersize=1, color="red", alpha=0.5)
gdf_sipongi_3857.plot(ax=axes[1], markersize=4, color="navy", alpha=0.7)
axes[1].set_title("Sesudah: EPSG:3857 (Web Mercator, meter)")
axes[1].set_xlabel("X (m)"); axes[1].set_ylabel("Y (m)")
axes[1].ticklabel_format(style="sci", scilimits=(0, 0))

if gdf_indonesia is not None:
    gdf_indonesia.to_crs(4326).boundary.plot(ax=axes[0], color="grey", linewidth=0.3)
    gdf_indonesia.to_crs(3857).boundary.plot(ax=axes[1], color="grey", linewidth=0.3)

legenda = [Line2D([0], [0], marker="o", ls="", color="red", label="Hotspot NASA (high)"),
           Line2D([0], [0], marker="o", ls="", color="navy", label="Hotspot SiPongi")]
for ax in axes:
    ax.legend(handles=legenda, loc="lower left")
plt.tight_layout()
plt.show()

**Interpretasi Tugas 2.** Setelah transformasi, nilai koordinat berubah dari satuan derajat menjadi meter (misalnya bujur ±110° menjadi X ±12 juta meter), namun **pola sebaran titik tetap sama** — menandakan proses reproyeksi berjalan benar. Karena wilayah Indonesia berada di sekitar ekuator, distorsi bentuk akibat proyeksi Mercator relatif kecil sehingga kedua peta tampak serupa.

> **Catatan teknis.** EPSG:3857 cocok untuk visualisasi dan *basemap* web, tetapi bukan proyeksi *equal-area* maupun *equidistant*. Faktor skala Mercator adalah 1/cos(lintang); pada lintang ±6° distorsinya hanya sekitar 0,5%. Untuk pengukuran jarak/luas yang presisi, disarankan menggunakan proyeksi UTM lokal (mis. `gdf.estimate_utm_crs()`).

## 5. TUGAS 3 — Operasi Geometri (Buffer 500 Meter)

*Buffer* menghasilkan poligon yang mencakup seluruh area dalam radius tertentu dari suatu geometri. Di sini setiap titik hotspot diberi *buffer* **500 meter** untuk merepresentasikan **zona bahaya/dampak 500 m dari titik api**.

Operasi *buffer* **wajib** dilakukan pada CRS metrik (EPSG:3857 dari Tugas 2) agar argumen jarak `500` diartikan sebagai 500 meter, bukan 500 derajat.

Karena 500 m sangat kecil pada skala nasional, visualisasi perbandingan difokuskan pada **sel grid 0,25° dengan kepadatan hotspot tertinggi**.

In [ ]:
JARAK_BUFFER_M = 500

# Buffer disimpan pada variabel baru (geometri berubah dari Point -> Polygon)
gdf_buffer_3857 = gdf_hotspot_3857.copy()
gdf_buffer_3857["geometry"] = gdf_hotspot_3857.geometry.buffer(JARAK_BUFFER_M, resolution=16)
gdf_buffer_3857["zona"] = f"Zona bahaya {JARAK_BUFFER_M} m"

print("Tipe geometri sebelum :", gdf_hotspot_3857.geom_type.unique().tolist())
print("Tipe geometri sesudah :", gdf_buffer_3857.geom_type.unique().tolist())
print(f"Luas 1 buffer (rata-rata)   : {gdf_buffer_3857.area.mean():,.0f} m² "
      f"(≈ {gdf_buffer_3857.area.mean() / 1e4:.1f} ha)")
print(f"Luas teoretis π·r²          : {np.pi * JARAK_BUFFER_M**2:,.0f} m²")
gdf_buffer_3857[["acq_date", "frp", "zona", "geometry"]].head()

In [ ]:
# Menentukan area fokus: sel grid 0,25° dengan jumlah hotspot terbanyak
UKURAN_SEL = 0.25
lat_sel = np.floor(gdf_hotspot.geometry.y / UKURAN_SEL) * UKURAN_SEL
lon_sel = np.floor(gdf_hotspot.geometry.x / UKURAN_SEL) * UKURAN_SEL
lat0, lon0 = pd.Series(list(zip(lat_sel.round(2), lon_sel.round(2)))).value_counts().idxmax()

area_fokus = gpd.GeoDataFrame(
    geometry=[box(lon0, lat0, lon0 + UKURAN_SEL, lat0 + UKURAN_SEL)], crs=4326).to_crs(3857)

mask_fokus = gdf_hotspot_3857.intersects(area_fokus.geometry.iloc[0])
pts_fokus  = gdf_hotspot_3857[mask_fokus]
buf_fokus  = gdf_buffer_3857[mask_fokus]

print(f"Area fokus: lintang {lat0:.2f}° s.d. {lat0 + UKURAN_SEL:.2f}°, "
      f"bujur {lon0:.2f}° s.d. {lon0 + UKURAN_SEL:.2f}°")
print(f"Jumlah hotspot di area fokus: {len(pts_fokus):,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7.5), sharex=True, sharey=True)

# SEBELUM: titik asli
pts_fokus.plot(ax=axes[0], color="red", markersize=10)
axes[0].set_title(f"SEBELUM — Titik Hotspot (n = {len(pts_fokus):,})")

# SESUDAH: buffer 500 m (di-overlay dengan titik asli)
buf_fokus.plot(ax=axes[1], color="orange", alpha=0.35, edgecolor="darkorange", linewidth=0.6)
pts_fokus.plot(ax=axes[1], color="red", markersize=3)
axes[1].set_title(f"SESUDAH — Buffer {JARAK_BUFFER_M} m (Zona Bahaya)")

for ax in axes:
    area_fokus.boundary.plot(ax=ax, color="grey", linestyle="--", linewidth=1)
    ax.set_xlabel("X (m) — EPSG:3857")
    ax.ticklabel_format(style="plain")
    ax.tick_params(axis="x", rotation=30)
axes[0].set_ylabel("Y (m) — EPSG:3857")

axes[1].legend(handles=[
    Line2D([0], [0], marker="o", ls="", color="red", label="Titik hotspot"),
    Patch(facecolor="orange", alpha=0.35, edgecolor="darkorange", label=f"Buffer {JARAK_BUFFER_M} m"),
    Line2D([0], [0], color="grey", ls="--", label="Batas area fokus"),
], loc="lower left")

fig.suptitle("Perbandingan Titik Hotspot dan Zona Bahaya 500 m", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Analisis luas zona bahaya: penjumlahan individual vs gabungan (dissolve/union)
luas_individu_km2 = buf_fokus.area.sum() / 1e6
zona_gabungan     = buf_fokus.dissolve()            # menggabungkan buffer yang saling tumpang tindih
luas_union_km2    = zona_gabungan.area.iloc[0] / 1e6

# Koreksi faktor skala Mercator & pembanding pada proyeksi UTM
lat_tengah = lat0 + UKURAN_SEL / 2
faktor     = np.cos(np.radians(lat_tengah))
crs_utm    = pts_fokus.estimate_utm_crs()
luas_utm_km2 = pts_fokus.to_crs(crs_utm).buffer(JARAK_BUFFER_M, resolution=16).union_all().area / 1e6

print(f"Luas total buffer (dijumlahkan)         : {luas_individu_km2:,.2f} km²")
print(f"Luas zona bahaya gabungan (EPSG:3857)   : {luas_union_km2:,.2f} km²")
print(f"Proporsi tumpang tindih                 : {(1 - luas_union_km2 / luas_individu_km2):.1%}")
print(f"Radius efektif di permukaan bumi        : {JARAK_BUFFER_M * faktor:.1f} m (lintang {lat_tengah:.2f}°)")
print(f"Luas zona bahaya gabungan (UTM {crs_utm.to_epsg()})  : {luas_utm_km2:,.2f} km²")

**Interpretasi Tugas 3.** Setiap titik hotspot kini menjadi poligon lingkaran beradius 500 m (≈ 78,5 ha per titik). Pada area dengan kepadatan tinggi, *buffer* saling tumpang tindih sehingga luas zona bahaya riil harus dihitung dari hasil penggabungan (`dissolve`/`union_all`), bukan dari penjumlahan luas tiap *buffer*. Zona ini dapat dimanfaatkan, misalnya, untuk mengidentifikasi permukiman, jalan, atau fasilitas publik yang berada dalam jangkauan dampak api dan asap.

> Karena data berada dekat ekuator, selisih antara *buffer* pada EPSG:3857 dan UTM sangat kecil. Namun untuk wilayah lintang tinggi, *buffer* sebaiknya dibuat pada proyeksi UTM agar radius 500 m benar-benar sesuai di lapangan.

## 6. TUGAS 4 — Visualisasi Tematik (*Choropleth Map*)

Peta *choropleth* mewarnai setiap poligon wilayah berdasarkan nilai suatu variabel kuantitatif. Di sini variabel yang digunakan adalah **`jumlah_hotspot` per provinsi dari data SiPongi** (hasil rekap pada bagian 2.2), dengan skema warna berurutan (*sequential*) **`OrRd`** yang sesuai dengan tema api/kebakaran (rendah = terang, tinggi = merah tua).

**Sumber poligon:**
- Jika berkas `peta_indonesia.shp` tersedia, poligon batas provinsi asli digunakan dan digabung (*merge*) dengan data SiPongi berdasarkan nama provinsi (sesuaikan `KOLOM_NAMA_PROV`).
- Jika tidak tersedia, dibuat **GeoDataFrame *dummy*** berisi 5 poligon persegi yang **mengaproksimasi secara kasar** letak 5 provinsi dengan hotspot terbanyak. Nilai numeriknya tetap diambil dari data SiPongi yang sebenarnya.

In [ ]:
KOLOM_NAMA_PROV = "PROVINSI"     # sesuaikan dengan nama kolom pada shapefile batas provinsi

if PATH_SHP_INDONESIA.exists():
    gdf_prov = gpd.read_file(PATH_SHP_INDONESIA).to_crs(4326)
    gdf_prov["kunci"] = gdf_prov[KOLOM_NAMA_PROV].astype(str).str.upper().str.strip()
    df_kunci = df_prov_sipongi.assign(kunci=df_prov_sipongi["provinsi"].str.upper().str.strip())
    gdf_choro = gdf_prov.merge(df_kunci[["kunci", "jumlah_hotspot"]], on="kunci", how="left")
    gdf_choro["provinsi"] = gdf_prov[KOLOM_NAMA_PROV]
    sumber_poligon = "Shapefile batas provinsi"
else:
    # Dummy: poligon persegi (aproksimasi kasar bounding box) — minx, miny, maxx, maxy
    bbox_dummy = {
        "Sulawesi Tenggara":   (121.3,  -6.2, 124.1, -3.0),
        "Sulawesi Selatan":    (119.0,  -5.8, 121.2, -2.0),
        "Kalimantan Timur":    (113.8,  -2.5, 119.0,  2.6),
        "Maluku":              (125.9,  -4.6, 131.0, -2.7),
        "Nusa Tenggara Timur": (118.9, -11.0, 125.2, -8.0),
    }
    gdf_choro = gpd.GeoDataFrame(
        {"provinsi": list(bbox_dummy.keys())},
        geometry=[box(*v) for v in bbox_dummy.values()],
        crs="EPSG:4326",
    ).merge(df_prov_sipongi, on="provinsi", how="left")
    sumber_poligon = "Poligon dummy (aproksimasi)"

gdf_choro["jumlah_hotspot"] = gdf_choro["jumlah_hotspot"].fillna(0).astype(int)
print("Sumber poligon:", sumber_poligon, "| CRS:", gdf_choro.crs)
gdf_choro[["provinsi", "jumlah_hotspot", "geometry"]].sort_values("jumlah_hotspot", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7.5))

gdf_choro.plot(
    column="jumlah_hotspot",
    cmap="OrRd",
    edgecolor="black",
    linewidth=0.8,
    legend=True,
    legend_kwds={"label": "Jumlah hotspot (SiPongi)", "orientation": "vertical", "shrink": 0.7},
    missing_kwds={"color": "lightgrey", "label": "Tidak ada data"},
    ax=ax,
)

# Label nama & nilai (hanya bila jumlah poligon sedikit agar tidak bertumpuk)
if len(gdf_choro) <= 10:
    for _, r in gdf_choro.iterrows():
        p = r.geometry.representative_point()
        ax.annotate(f"{r['provinsi']}\n{r['jumlah_hotspot']:,}", xy=(p.x, p.y),
                    ha="center", va="center", fontsize=9, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7, lw=0))

ax.set_title(f"Peta Choropleth Jumlah Hotspot per Provinsi — {sumber_poligon}",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Bujur (°)")
ax.set_ylabel("Lintang (°)")
ax.annotate("Sumber: SiPongi KLHK", xy=(0.99, 0.01), xycoords="axes fraction",
            ha="right", fontsize=8, color="grey")
plt.tight_layout()
plt.show()

**Interpretasi Tugas 4.** Gradasi warna `OrRd` memudahkan pembacaan intensitas: semakin merah tua suatu wilayah, semakin banyak hotspot yang terdeteksi. *Legend* berupa *colorbar* menjelaskan rentang nilai yang diwakili tiap warna. Skema *sequential* dipilih karena variabel `jumlah_hotspot` bersifat kuantitatif dan berurutan dari rendah ke tinggi.

> **Catatan.** Bila menggunakan poligon *dummy*, bentuk dan luas wilayah **tidak** merepresentasikan batas administrasi sebenarnya — poligon hanya digunakan untuk mendemonstrasikan teknik *choropleth*. Untuk analisis resmi, gunakan batas provinsi dari sumber otoritatif (mis. Badan Informasi Geospasial). Untuk perbandingan antarwilayah yang adil, pertimbangkan juga normalisasi (mis. hotspot per 1.000 km²).

## 7. Kesimpulan

Sel berikut merangkum angka-angka kunci hasil analisis.

In [ ]:
print("RINGKASAN ANALISIS")
print("=" * 60)
print(f"Hotspot NASA setelah pembersihan  : {len(df_nasa):,} titik "
      f"({df_nasa['acq_date'].min():%d-%m-%Y} s.d. {df_nasa['acq_date'].max():%d-%m-%Y})")
print(f"  - confidence tinggi (GeoJSON)   : {len(gdf_hotspot):,} titik")
print(f"Bulan dengan hotspot terbanyak    : {hitung_bulan.sum(axis=1).idxmax()} "
      f"({hitung_bulan.sum(axis=1).max():,} titik)")
print(f"Total hotspot SiPongi (rekap)     : {df_prov_sipongi['jumlah_hotspot'].sum():,}")
print(f"Provinsi tertinggi (SiPongi)      : {df_prov_sipongi.iloc[0]['provinsi']} "
      f"({df_prov_sipongi.iloc[0]['jumlah_hotspot']:,})")
print(f"Zona bahaya 500 m di area fokus   : {luas_union_km2:,.2f} km² dari {len(pts_fokus):,} titik")

Secara ringkas, notebook ini telah:

1. **Memuat dan membersihkan** data hotspot NASA FIRMS (gabungan *archive* dan NRT) serta data rekap SiPongi, termasuk penyaringan deteksi non-vegetasi dan deteksi berkepercayaan rendah.
2. **Tugas 1** — membaca data spasial dengan `gpd.read_file()`, menampilkan `.head()`, memeriksa `.crs` (EPSG:4326, satuan derajat), dan memvisualisasikan sebaran titik dengan `.plot()`.
3. **Tugas 2** — memastikan CRS dengan `.set_crs(epsg=4326)` bila belum ada, lalu memproyeksikan ulang ke **EPSG:3857** dengan `.to_crs()`; pola sebaran tetap konsisten sementara satuan berubah menjadi meter.
4. **Tugas 3** — membuat *buffer* 500 m sebagai zona bahaya, membandingkan kondisi sebelum–sesudah, dan menghitung luas zona gabungan dengan memperhitungkan tumpang tindih.
5. **Tugas 4** — menyusun peta *choropleth* jumlah hotspot per provinsi dengan skema warna berurutan `OrRd` dan *legend*.

**Saran pengembangan:** (a) menggunakan batas administrasi resmi untuk *spatial join* titik NASA ke provinsi (`gpd.sjoin`) sehingga kedua sumber data dapat dibandingkan pada periode yang sama; (b) menambahkan *basemap* dengan pustaka `contextily` (memanfaatkan data EPSG:3857); (c) melakukan analisis kepadatan (*kernel density*) atau klasterisasi spasial (mis. DBSCAN) untuk mengidentifikasi klaster kebakaran.